## Local Accumulation / Reduction

In [1]:
import threading
import time
import numpy as np
from PIL import Image

In [ ]:
# Load the image
image_path = "test_images/large_1024x1024.png"
image = np.array(Image.open(image_path).convert("L"), dtype=np.float64)

In [3]:
gaussian_kernel = np.array([
    [1, 2, 1],
    [2, 4, 2],
    [1, 2, 1],
], dtype=np.float64,) / 16.0

In [4]:
# Safe Approach 1: Local Accumulation / Reduction (Pixel Sum)
def local_pixel_worker(local_results, worker_idx, partial_sum, repeats):
    local_acc = 0.0
    
    for _ in range(repeats):
        local_acc += partial_sum
    
    local_results[worker_idx] = local_acc


def run_local_output_pixel(n_workers=3, repeats=1000, verbose=True):
    i, j = image.shape[0] // 2, image.shape[1] // 2
    partial_sums = []

    for k in range(n_workers):
        partial = 0.0
        for l in range(3):
            partial += image[i + k - 1, j + l - 1] * gaussian_kernel[k, l]
        partial_sums.append(partial)

    local_results = [0.0] * n_workers
    threads = [
        threading.Thread(
            target=local_pixel_worker,
            args=(local_results, idx, partial, repeats),
        )
        for idx, partial in enumerate(partial_sums)
    ]

    start = time.perf_counter()
    for t in threads:
        t.start()
    for t in threads:
        t.join()

    # Reduction step after all threads finish
    actual = sum(local_results)
    elapsed = time.perf_counter() - start

    expected = sum(partial_sums) * repeats
    abs_error = abs(expected - actual)

    if verbose:
        print("Expected:", expected)
        print("Actual:", actual)
        print("Absolute error:", abs_error)
        print(f"Execution time: {elapsed:.4f}s")

    return {
        "expected": expected,
        "actual": actual,
        "abs_error": abs_error,
        "elapsed": elapsed,
    }
    
for run in range(5):
    print(f"\nRUN {run + 1} - Pixel (Local Accumulation)")
    run_local_output_pixel()


RUN 1 - Pixel (Local Accumulation)
Expected: 18062.5
Actual: 18062.5
Absolute error: 0.0
Execution time: 0.0014s

RUN 2 - Pixel (Local Accumulation)
Expected: 18062.5
Actual: 18062.5
Absolute error: 0.0
Execution time: 0.0007s

RUN 3 - Pixel (Local Accumulation)
Expected: 18062.5
Actual: 18062.5
Absolute error: 0.0
Execution time: 0.0005s

RUN 4 - Pixel (Local Accumulation)
Expected: 18062.5
Actual: 18062.5
Absolute error: 0.0
Execution time: 0.0004s

RUN 5 - Pixel (Local Accumulation)
Expected: 18062.5
Actual: 18062.5
Absolute error: 0.0
Execution time: 0.0004s


In [5]:
# Safe Approach 2: Local Accumulation / Reduction (Counter)
def local_counter_worker(local_counts, worker_idx, iterations):
    local_count = 0
    
    for _ in range(iterations):
        local_count += 1
        
    local_counts[worker_idx] = local_count


def run_local_counter(n_workers=2, iterations=50000, verbose=True):
    local_counts = [0] * n_workers
    
    threads = [
        threading.Thread(
            target=local_counter_worker,
            args=(local_counts, idx, iterations),
        )
        for idx in range(n_workers)
    ]

    start = time.perf_counter()
    for t in threads:
        t.start()
    for t in threads:
        t.join()

    # Reduction step after all threads finish
    actual = sum(local_counts)
    elapsed = time.perf_counter() - start

    expected = n_workers * iterations
    lost_updates = expected - actual

    if verbose:
        print("Expected:", expected)
        print("Actual:", actual)
        print("Lost updates:", lost_updates)
        print(f"Execution time: {elapsed:.4f}s")

    return {
        "expected": expected,
        "actual": actual,
        "lost_updates": lost_updates,
        "elapsed": elapsed,
    }
    
for run in range(5):
    print(f"\nRUN {run + 1} - Counter (Local Accumulation)")
    run_local_counter()


RUN 1 - Counter (Local Accumulation)
Expected: 100000
Actual: 100000
Lost updates: 0
Execution time: 0.0056s

RUN 2 - Counter (Local Accumulation)
Expected: 100000
Actual: 100000
Lost updates: 0
Execution time: 0.0060s

RUN 3 - Counter (Local Accumulation)
Expected: 100000
Actual: 100000
Lost updates: 0
Execution time: 0.0037s

RUN 4 - Counter (Local Accumulation)
Expected: 100000
Actual: 100000
Lost updates: 0
Execution time: 0.0036s

RUN 5 - Counter (Local Accumulation)
Expected: 100000
Actual: 100000
Lost updates: 0
Execution time: 0.0032s
